# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVR

## 2. Datos

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [3]:
# Ver las primeras filas para entender qué tenemos
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [4]:
# Cuántas filas y columnas hay
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])

Filas: 912
Columnas: 13


In [5]:
# Tipo de cada columna y valores nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    object 
 2   Product           912 non-null    object 
 3   TypeName          912 non-null    object 
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    object 
 6   Cpu               912 non-null    object 
 7   Ram               912 non-null    object 
 8   Memory            912 non-null    object 
 9   Gpu               912 non-null    object 
 10  OpSys             912 non-null    object 
 11  Weight            912 non-null    object 
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), object(10)
memory usage: 92.8+ KB


In [6]:
# Estadísticas del precio (nuestra variable objetivo)
df['Price_in_euros'].describe()

count     912.000000
mean     1111.724090
std       687.959172
min       174.000000
25%       589.000000
50%       978.000000
75%      1483.942500
max      6099.000000
Name: Price_in_euros, dtype: float64

In [7]:
# Precio medio por marca — da intuición sobre los datos
df.groupby('Company')['Price_in_euros'].mean().sort_values(ascending=False)

Company
Razer        2987.333333
LG           2299.000000
Google       1879.000000
Microsoft    1716.970000
MSI          1713.425135
Apple        1540.322353
Huawei       1424.000000
Samsung      1423.000000
Toshiba      1276.558824
Xiaomi       1199.616667
Dell         1173.998680
HP           1093.484639
Asus         1068.684050
Lenovo       1053.578416
Fujitsu       769.000000
Acer          607.613514
Mediacom      294.333333
Chuwi         246.945000
Vero          235.400000
Name: Price_in_euros, dtype: float64

### 2.2 Definir X e y

Antes de definir X, tenemos que **procesar las columnas de texto** para convertirlas a números.
Los modelos de ML no entienden texto, solo números.

Columnas que necesitan transformación:
- `Ram` → viene como "8GB", hay que quedarse solo con el número
- `Weight` → viene como "1.86kg", hay que quedarse solo con el número  
- `Company`, `TypeName`, `OpSys` → texto de categorías, usamos LabelEncoder

In [8]:
# Convertir Ram: quitamos el texto "GB" y lo convertimos a entero
df['Ram_GB'] = df['Ram'].str.replace('GB', '').astype(int)

# Convertir Weight: quitamos el texto "kg" y lo convertimos a decimal
df['Weight_kg'] = df['Weight'].str.replace('kg', '').astype(float)

print('Ram antes:', df['Ram'].head(3).tolist())
print('Ram después:', df['Ram_GB'].head(3).tolist())
print()
print('Weight antes:', df['Weight'].head(3).tolist())
print('Weight después:', df['Weight_kg'].head(3).tolist())

Ram antes: ['8GB', '16GB', '8GB']
Ram después: [8, 16, 8]

Weight antes: ['1.86kg', '2.59kg', '2.04kg']
Weight después: [1.86, 2.59, 2.04]


In [9]:
# LabelEncoder convierte categorías de texto a números
# Ejemplo: Apple=0, Asus=1, Dell=2 ...
# IMPORTANTE: guardamos cada encoder en un diccionario para reutilizarlo en test.csv

les = {}  # diccionario para guardar los encoders

for col in ['Company', 'TypeName', 'OpSys']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    les[col] = le  # guardamos el encoder

print('Company (primeras 5):', df['Company'].head(5).tolist())
print('Company_enc (primeras 5):', df['Company_enc'].head(5).tolist())

Company (primeras 5): ['HP', 'Dell', 'HP', 'Apple', 'Dell']
Company_enc (primeras 5): [7, 4, 7, 1, 4]


In [10]:
# Definir las features (X) y el target (y)
# Usamos las columnas numéricas: las que acabamos de crear + Inches que ya era número
features = ['Ram_GB', 'Weight_kg', 'Inches', 'Company_enc', 'TypeName_enc', 'OpSys_enc']

X = df[features]
y = df['Price_in_euros']

print('Shape de X:', X.shape)
print('Shape de y:', y.shape)
X.head()

Shape de X: (912, 6)
Shape de y: (912,)


,Ram_GB,Weight_kg,Inches,Company_enc,TypeName_enc,OpSys_enc
0,8,1.86,15.6,7,3,5
1,16,2.59,15.6,4,1,5
2,8,2.04,15.6,7,3,5
3,8,1.34,13.3,1,4,8
4,4,2.25,15.6,4,3,2


### 2.3 Dividir en train y test

In [11]:
# Dividimos los datos: 80% para entrenar, 20% para validar localmente
# random_state=42 asegura que siempre obtenemos el mismo split (reproducibilidad)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print('Tamaño X_train:', X_train.shape)
print('Tamaño X_val:', X_val.shape)

Tamaño X_train: (729, 6)
Tamaño X_val: (183, 6)


## 3. Procesado de datos
> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [12]:
# En este caso no usamos scaler porque RandomForest no lo necesita.
# El procesado ya lo hicimos arriba (Ram_GB, Weight_kg, LabelEncoders).
# Las features ya son todas numéricas, así que podemos pasar directamente al modelo.

print('Features utilizadas:', features)
print('Tipos de datos:')
print(X_train.dtypes)

Features utilizadas: ['Ram_GB', 'Weight_kg', 'Inches', 'Company_enc', 'TypeName_enc', 'OpSys_enc']
Tipos de datos:
Ram_GB            int64
Weight_kg       float64
Inches          float64
Company_enc       int64
TypeName_enc      int64
OpSys_enc         int64
dtype: object


## 4. Modelado

### 4.1 Entrenamiento

In [13]:
# Usamos RandomForestRegressor: crea muchos árboles de decisión y promedia sus predicciones
# n_estimators=100 → 100 árboles
# random_state=42  → reproducibilidad

modelo = RandomForestRegressor(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

print('Modelo entrenado ✅')

Modelo entrenado ✅


### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [14]:
# Predecimos sobre el conjunto de validación
y_pred = modelo.predict(X_val)

# Calculamos el RMSE
rmse = root_mean_squared_error(y_val, y_pred)
print(f'RMSE en validación: {rmse:.2f} €')
print(f'Interpretación: el modelo se equivoca de media {rmse:.0f}€ en sus predicciones')

RMSE en validación: 414.95 €
Interpretación: el modelo se equivoca de media 415€ en sus predicciones


### 4.3 Optimización (up to you 🫰🏻)

In [15]:
# Probamos con más árboles a ver si mejora el RMSE
modelo_v2 = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42)
modelo_v2.fit(X_train, y_train)

y_pred_v2 = modelo_v2.predict(X_val)
rmse_v2 = root_mean_squared_error(y_val, y_pred_v2)
print(f'RMSE modelo v2 (200 árboles): {rmse_v2:.2f} €')

# Nos quedamos con el mejor
if rmse_v2 < rmse:
    modelo_final = modelo_v2
    print('Usaremos el modelo v2 (es mejor)')
else:
    modelo_final = modelo
    print('Usaremos el modelo original (es mejor o igual)')

RMSE modelo v2 (200 árboles): 414.93 €
Usaremos el modelo v2 (es mejor)


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [16]:
# Reentrenamos con X e y completos (no el split)
modelo_final.fit(X, y)

print('Modelo reentrenado con el 100% de los datos de train ✅')

Modelo reentrenado con el 100% de los datos de train ✅


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [17]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [18]:
# Exactamente las mismas transformaciones que hicimos en train

# 1. Ram y Weight numéricos
X_pred['Ram_GB']    = X_pred['Ram'].str.replace('GB', '').astype(int)
X_pred['Weight_kg'] = X_pred['Weight'].str.replace('kg', '').astype(float)

# 2. LabelEncoder: usamos .transform() (NO .fit_transform()) con los encoders ya entrenados
for col in ['Company', 'TypeName', 'OpSys']:
    X_pred[col + '_enc'] = les[col].transform(X_pred[col])

# 3. Seleccionar las mismas features que usamos en train
X_pred_final = X_pred[features]

print('Test procesado ✅')
print('Shape:', X_pred_final.shape)
X_pred_final.head()

Test procesado ✅
Shape: (391, 6)


,Ram_GB,Weight_kg,Inches,Company_enc,TypeName_enc,OpSys_enc
0,16,2.400,15.6,10,1,4
1,4,2.400,15.6,0,3,2
2,4,1.900,15.6,10,3,4
3,8,2.191,15.6,4,0,5
4,4,1.950,14.0,7,3,5


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [19]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [20]:
# Predecir los precios del test
predicciones = modelo_final.predict(X_pred_final)

# Crear el dataframe de submission con el formato que pide Kaggle
submission = sample.copy()                      # copiamos el sample para mantener los IDs en orden
submission['Price_in_euros'] = predicciones     # añadimos nuestras predicciones

print('Predicciones generadas ✅')
print(f'Min: {predicciones.min():.0f}€ | Max: {predicciones.max():.0f}€ | Media: {predicciones.mean():.0f}€')
submission.head(10)

Predicciones generadas ✅
Min: 240€ | Max: 4844€ | Media: 1159€


,laptop_ID,Price_in_euros
0,209,1295.662133
1,1281,353.157450
2,1168,382.164383
3,1231,881.705471
4,1020,1119.019396
5,379,649.651190
6,553,684.254028
7,172,866.500000
8,779,1058.479767
9,609,366.092367


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [ ]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" Perfecto, submission guardada como '{filename}'.")

In [22]:
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260625_121546.csv'. ¡A Kaggle!
